# DRGU PyTorch Implementation Example

This notebook demonstrates the PyTorch implementation of DRGU (Doubly Robust Generalized U) and compares it with the JAX implementation.


In [1]:
import torch
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
import sys
import os

# Add path to import robustinfer
# sys.path.append('../../python_lib/src')
from robustinfer import DRGU, DRGUJax



## Generate Simulation Data


In [2]:
# Generate data with lognormal errors
np.random.seed(42)
n = 200

# Generate covariates
x1 = np.random.normal(0, 1, n)

# Generate treatment (binary)
treatment_prob = 1 / (1 + np.exp(-(0.5 + 0.3 * x1)))
x = np.random.binomial(1, treatment_prob, n)

# Generate response
y = 0.5 + 2.0 * x + 1.0 * x1 + np.random.lognormal(0, 0.5, n)

print(f"Data shapes - y: {y.shape}, x: {x.shape}, x1: {x1.shape}")

# Create DataFrame
df = pd.DataFrame({
    "y": y,
    "x": x,
    "w1": x1
})

print("\nDataFrame head:")
df.head()


Data shapes - y: (200,), x: (200,), x1: (200,)

DataFrame head:


,y,x,w1
0,1.943786,0,0.496714
1,1.584184,0,-0.138264
2,2.559780,0,0.647689
3,4.841261,1,1.523030
4,3.384411,1,-0.234153


## Compare JAX and PyTorch Implementations

### JAX Implementation


In [3]:
# Fit JAX model
print("Fitting JAX model...")
model_jax = DRGUJax(df, covariates=["w1"], treatment="x", response="y")
model_jax.fit()

print("JAX Model Results:")
print(f"Coefficients: {model_jax.coefficients}")
print(f"Variance matrix shape: {model_jax.variance_matrix.shape}")

# JAX model summary
jax_summary = model_jax.summary()
print("\nJAX Model Summary:")
jax_summary


Fitting JAX model...
Step 0 gradient norm: 0.3638095259666443
Step 10 gradient norm: 3.6998678751842817e-06
converged after 12 iterations
JAX Model Results:
Coefficients: [ 0.9093212   0.40332583  0.27669135  4.786469    2.5357916  -1.9200606 ]
Variance matrix shape: (6, 6)

JAX Model Summary:


,Names,Coefficient,Null_Hypothesis,Std_Error,Z_Score,P_Value
0,delta,0.909321,0.5,0.015350,26.666155,0.000000e+00
1,beta_0,0.403326,0.0,0.144742,2.786525,5.327651e-03
2,beta_1,0.276691,0.0,0.158714,1.743330,8.127601e-02
3,gamma_0,4.786469,0.0,1.224857,3.907778,9.314873e-05
4,gamma_1,2.535792,0.0,1.022779,2.479315,1.316350e-02
5,gamma_2,-1.920061,0.0,0.368133,-5.215668,1.831558e-07


### PyTorch Implementation


In [4]:
# Fit PyTorch model on CPU
print("Fitting PyTorch model (CPU)...")
model_torch_cpu = DRGU(df, covariates=["w1"], treatment="x", response="y", device='cpu')
model_torch_cpu.fit()

print("PyTorch (CPU) Model Results:")
print(f"Coefficients: {model_torch_cpu.coefficients}")
print(f"Variance matrix shape: {model_torch_cpu.variance_matrix.shape}")

# PyTorch model summary
torch_summary = model_torch_cpu.summary()
print("\nPyTorch (CPU) Model Summary:")
torch_summary


Fitting PyTorch model (CPU)...
Step 0 gradient norm: 0.363810
Step 10 gradient norm: 0.000002
Converged after 12 iterations
PyTorch (CPU) Model Results:
Coefficients: tensor([ 0.9093,  0.4033,  0.2767,  4.7865,  2.5358, -1.9201])
Variance matrix shape: torch.Size([6, 6])

PyTorch (CPU) Model Summary:


,Names,Coefficient,Null_Hypothesis,Std_Error,Z_Score,P_Value
0,delta,0.909321,0.5,0.015350,26.666153,0.000000e+00
1,beta_0,0.403326,0.0,0.144741,2.786526,5.327635e-03
2,beta_1,0.276691,0.0,0.158714,1.743330,8.127597e-02
3,gamma_0,4.786469,0.0,1.224850,3.907801,9.314010e-05
4,gamma_1,2.535791,0.0,1.022775,2.479326,1.316310e-02
5,gamma_2,-1.920060,0.0,0.368128,-5.215736,1.830884e-07


In [5]:
# Compare coefficients
jax_coeffs = np.array(model_jax.coefficients)
torch_coeffs = model_torch_cpu.coefficients.cpu().numpy()

print("Coefficient Comparison:")
comparison_df = pd.DataFrame({
    'Parameter': jax_summary['Names'],
    'JAX': jax_coeffs,
    'PyTorch': torch_coeffs,
    'Difference': np.abs(jax_coeffs - torch_coeffs),
    'Relative_Error': np.abs(jax_coeffs - torch_coeffs) / (np.abs(jax_coeffs) + 1e-8)
})

comparison_df


Coefficient Comparison:


,Parameter,JAX,PyTorch,Difference,Relative_Error
0,delta,0.909321,0.909321,0.000000e+00,0.000000e+00
1,beta_0,0.403326,0.403326,0.000000e+00,0.000000e+00
2,beta_1,0.276691,0.276691,2.980232e-08,1.077096e-07
3,gamma_0,4.786469,4.786469,0.000000e+00,0.000000e+00
4,gamma_1,2.535792,2.535791,2.384186e-07,9.402136e-08
5,gamma_2,-1.920061,-1.920060,2.384186e-07,1.241724e-07
